In [1]:
# ============================================================
# EV Range Study (2023) — Base case + charging-frequency scenarios
# - BTS ingest (national monthly trips by distance)
# - Single set of bin-average distances (across months) via bounded LSQ
# - Tables in billions; VMT = Trips(B) * Avg miles (display-exact)
# - Base EV allocation: round-trip range constraint; no mid-trip charging
# - Scenarios: charges/week from doc; weights from ACS B25032
# ============================================================

# ---------------- CONFIG ----------------
YEAR = 2023
MONTH_TO_SHOW = 12                  # 1..12 for Table 1 (monthly)
ROUND_TRIP = True                   # EV must cover out+back (base case)
RANGES = [50, 75, 100, 125, 150]    # EV ranges to evaluate (miles)
OUTDIR = "Definitions/EV_VMT_Results"

# Emissions & costs
MPG_GAS = 26.2
CO2_PER_GAL = 8887.0               # g CO2/gal
EV_KWH_PER_MI = 0.29               # kWh/mi
GRID_G_PER_KWH = 387.0             # g/kWh
ETA_MI_PER_KWH = 3.6               # mi/kWh (battery sizing)
PACK_COST_PER_KWH = 115.0          # $/kWh
AVG_DRIVER_MILES = 13496.0         # avg miles/vehicle/year

# FHWA monthly VMT (billions) for 2023 (targets)
FHWA_VMT_B = {
    1: 248.9, 2: 235.4, 3: 273.7, 4: 258.2, 5: 289.6, 6: 285.6,
    7: 289.8, 8: 290.9, 9: 278.0, 10: 284.0, 11: 265.1, 12: 263.6
}

# Toggle: force each month to match FHWA exactly (scales monthly VMT after fitting x_opt)
NORMALIZE_TO_FHWA = True

# Data sources
BTS_URL  = "https://data.bts.gov/resource/w96p-f2qv.json"
LEVEL_OK = {"national", "nation", "n"}     # accepted Geographic Level values (case-insensitive)

# ACS: build scenario weights from Tenure x Units
USE_ACS = True
ACS_VARS_URL = "https://api.census.gov/data/2024/acs/acs1/variables.json"
ACS_DATA_URL = "https://api.census.gov/data/2024/acs/acs1"
ACS_TABLE    = "B25032"  # Tenure by Units in Structure (1-year)

# If you prefer to bypass ACS, fill this dict and set USE_ACS=False.
# It should be weights over the *charging levels* below (must sum to 1.0).
LEVEL_WEIGHT_OVERRIDE = {
    # "everyday": 0.00,
    # "garage":   0.00,
    # "driveway": 0.00,
    # "work":     0.00,
    # "weekly":   0.00,
    # "errands":  0.00,
    # "none":     0.00,
}

# ---------------- IMPORTS ----------------
import os, re, requests, math, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs(OUTDIR, exist_ok=True)

# SciPy bounded LSQ if available
try:
    from scipy.optimize import lsq_linear
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

# ---------------- HELPERS ----------------
def get_json(url, params=None, timeout=60):
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def lb(lbl: str) -> int:
    """Numeric sort for trip bins '0–1', '1–3', ... '500+'."""
    if lbl == "0–1": return 0
    m = re.match(r"(\d+)", lbl)
    return int(m.group(1)) if m else 9999

def bin_bounds(lbl: str):
    """Bounds for the average distance in each BTS bin (miles)."""
    bounds = {
        "0–1": (0.25, 1.0),
        "1–3": (1.0, 3.0),
        "3–5": (3.0, 5.0),
        "5–10": (5.0, 10.0),
        "10–25": (10.0, 25.0),
        "25–50": (25.0, 50.0),
        "50–100": (50.0, 100.0),
        "100–250": (100.0, 250.0),
        "250–500": (250.0, 500.0),
        "500+": (500.0, 1000.0),
    }
    return bounds[lbl]

def build_friendly(year: int) -> pd.DataFrame:
    """Pull BTS YEAR national monthly sums by distance bin -> `friendly` (12 x bins)."""
    sample = get_json(BTS_URL, {"$limit": 1})
    if not sample:
        raise RuntimeError("BTS endpoint returned 0 rows.")
    cols = list(sample[0].keys())

    COL = {
        "level": "level",
        "date": "date",
        "month": "month",
        "trips_<1": "trips_1",
        "trips_1_3": "trips_1_3",
        "trips_3_5": "trips_3_5",
        "trips_5_10": "trips_5_10",
        "trips_10_25": "trips_10_25",
        "trips_25_50": "trips_25_50",
        "trips_50_100": "trips_50_100",
        "trips_100_250": "trips_100_250",
        "trips_250_500": "trips_250_500",
        "trips_500+": "trips_500",
    }
    trip_cols = [v for k, v in COL.items() if k.startswith("trips_") and v in cols]

    BIN_LABEL = {
        "trips_1": "0–1", "trips_1_3": "1–3", "trips_3_5": "3–5", "trips_5_10": "5–10",
        "trips_10_25": "10–25", "trips_25_50": "25–50", "trips_50_100": "50–100",
        "trips_100_250": "100–250", "trips_250_500": "250–500", "trips_500": "500+",
    }

    has_month = COL["month"] in cols
    date_col  = COL["date"]
    level_col = COL["level"] if COL["level"] in cols else None

    if has_month:
        select_parts = [COL["month"]]
        groupby = orderby = COL["month"]
    else:
        select_parts = [f"date_trunc_ym({date_col}) as ym"]
        groupby = orderby = "ym"

    select_parts += [f"sum({c}) as {c}" for c in trip_cols]
    where_parts = [
        f"{date_col} >= '{year}-01-01T00:00:00.000'",
        f"{date_col} <  '{year+1}-01-01T00:00:00.000'",
    ]
    if level_col:
        where_parts.append(f"lower({level_col}) in ({', '.join(repr(x) for x in LEVEL_OK)})")

    params = {
        "$select": ", ".join(select_parts),
        "$where": " AND ".join(where_parts),
        "$group": groupby,
        "$order": orderby,
        "$limit": 5000
    }
    rows = get_json(BTS_URL, params)
    if not rows:
        params2 = params.copy()
        where2  = [w for w in where_parts if "lower(" not in w]
        params2["$where"] = " AND ".join(where2)
        rows = get_json(BTS_URL, params2)
        if not rows:
            raise RuntimeError("BTS aggregation returned 0 rows.")

    df = pd.DataFrame(rows)
    if has_month:
        df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
        df["year"]  = year
    else:
        df["ym"]    = pd.to_datetime(df["ym"], errors="coerce").dt.tz_localize(None)
        df["month"] = df["ym"].dt.month
        df["year"]  = df["ym"].dt.year

    years = df["year"].dropna().unique().tolist()
    assert years == [year], f"Pulled years are {years}, expected [{year}]"

    for c in trip_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    friendly = df[["month"] + trip_cols].rename(columns=BIN_LABEL)
    friendly = friendly.sort_values("month").reset_index(drop=True)
    friendly = friendly.reindex(columns=["month"] + sorted([c for c in friendly.columns if c != "month"], key=lb))
    return friendly

# ---------------- PART 1: BTS ingest & shared x via bounded LSQ ----------------
friendly = build_friendly(YEAR)
bins = [c for c in friendly.columns if c != "month"]
A_raw = friendly[bins].to_numpy(float)        # trips per month (counts), shape (12,B)
months = friendly["month"].tolist()
y_B = np.array([FHWA_VMT_B[m] for m in months], float)   # FHWA target in billions
y_miles = y_B * 1e9

lo = np.array([bin_bounds(b)[0] for b in bins], float)
hi = np.array([bin_bounds(b)[1] for b in bins], float)

if _HAVE_SCIPY:
    res = lsq_linear(A_raw, y_miles, bounds=(lo, hi), lsmr_tol='auto', verbose=0)
    x_opt = res.x
else:
    # Fallback: unconstrained LSQ then clip to bounds (simple projection)
    x_uncon, *_ = np.linalg.lstsq(A_raw, y_miles, rcond=None)
    x_opt = np.clip(x_uncon, lo, hi)

# Monthly modeled VMT using the same x across months (pre-normalization)
model_monthly_miles = A_raw @ x_opt
model_monthly_B_pre = model_monthly_miles / 1e9
err_pct_pre = 100 * (model_monthly_B_pre - y_B) / y_B

monthly_summary = pd.DataFrame({
    "Month": months,
    "FHWA VMT (B)": np.round(y_B, 3),
    "Model VMT (B) (pre-norm)": np.round(model_monthly_B_pre, 3),
    "Error (%) (pre-norm)": np.round(err_pct_pre, 2),
})
display(monthly_summary)
print("Max |Error| % (pre-normalization):", np.max(np.abs(err_pct_pre)).round(2))

# ---------------- OPTIONAL: per-month normalization to match FHWA exactly ----------------
# Keeps x_opt identical across months; scales each month’s VMT to FHWA total.
vmt_miles = (A_raw * x_opt.reshape(1, -1))   # (12, B), miles (pre-norm)
if NORMALIZE_TO_FHWA:
    s = (y_B / np.maximum((vmt_miles.sum(axis=1) / 1e9), 1e-12)).reshape(-1, 1)
    vmt_miles *= s
    model_monthly_B_post = (vmt_miles.sum(axis=1) / 1e9)
    err_pct_post = 100 * (model_monthly_B_post - y_B) / y_B
    print("Max |Error| % (post-normalization):", np.max(np.abs(err_pct_post)).round(6))

# ---------------- PART 1C: Table 1 (monthly) — display-consistent ----------------
row_trips_counts     = friendly.loc[friendly["month"].eq(MONTH_TO_SHOW), bins].iloc[0].to_numpy(float)
trips_B_per_bin_fp   = row_trips_counts / 1e9
trips_B_per_bin_disp = np.round(trips_B_per_bin_fp, 3)      # DISPLAYED
avg_miles_disp       = np.round(x_opt, 1)                    # DISPLAYED

# VMT (B) must be the product of the *displayed* fields:
vmt_B_per_bin_disp   = np.round(trips_B_per_bin_disp * avg_miles_disp, 3)  # DISPLAYED

table1_month = pd.DataFrame({
    "Trip Distance Bin (miles)": bins,
    "Trips (B/month)": trips_B_per_bin_disp,
    "Calculated Average Bin Distance (miles)": avg_miles_disp,
    "Calculated VMT (B)": vmt_B_per_bin_disp,
}).sort_values("Trip Distance Bin (miles)", key=lambda s: s.map(lb)).reset_index(drop=True)

# Row-level sanity check (DISPLAY equality: round the RHS to 3 dp too)
lhs = table1_month["Calculated VMT (B)"].to_numpy()
rhs = np.round(
    table1_month["Trips (B/month)"].to_numpy()
  * table1_month["Calculated Average Bin Distance (miles)"].to_numpy(), 3
)
assert np.array_equal(lhs, rhs), "Monthly Table 1: VMT != Trips(B)*Avg miles (display)."

display(table1_month)

# Full-precision monthly checks (independent of display rounding)
calc_fp_B_pre  = float((trips_B_per_bin_fp * x_opt).sum())
target_B       = float(y_B[months.index(MONTH_TO_SHOW)])
msg = (f"[Month {MONTH_TO_SHOW}] Pre-norm: VMT = {calc_fp_B_pre:.3f} B vs FHWA = {target_B:.1f} B  "
       f"-> Error = {100*(calc_fp_B_pre-target_B)/target_B:.2f}%")
if NORMALIZE_TO_FHWA:
    calc_post_B = float((vmt_miles[months.index(MONTH_TO_SHOW)].sum()) / 1e9)
    msg += f" | Post-norm: {calc_post_B:.3f} B (exact FHWA)"
print(msg)


# ---------------- PART 1D: Annual Table — display-consistent ----------------
annual_trips_counts  = A_raw.sum(axis=0)                 # counts/year per bin
annual_trips_B_fp    = annual_trips_counts / 1e9
annual_trips_B_disp  = np.round(annual_trips_B_fp, 3)    # DISPLAYED
annual_avg_miles_disp= np.round(x_opt, 1)                # DISPLAYED

annual_vmt_B_disp    = np.round(annual_trips_B_disp * annual_avg_miles_disp, 3)  # DISPLAYED

annual_table = pd.DataFrame({
    "Trip Distance Bin (miles)": bins,
    "Trips (B/year)": annual_trips_B_disp,
    "Average Bin Distance (miles)": annual_avg_miles_disp,
    "Calculated VMT (B/year)": annual_vmt_B_disp,
}).sort_values("Trip Distance Bin (miles)", key=lambda s: s.map(lb)).reset_index(drop=True)

# Row-level sanity check (DISPLAY equality: round the RHS to 3 dp too)
lhsA = annual_table["Calculated VMT (B/year)"].to_numpy()
rhsA = np.round(
    annual_table["Trips (B/year)"].to_numpy()
  * annual_table["Average Bin Distance (miles)"].to_numpy(), 3
)
assert np.array_equal(lhsA, rhsA), "Annual Table 1: VMT != Trips(B)*Avg miles (display)."

display(annual_table)

# Annual full-precision reconciliation (pre/post normalization report)
annual_model_B_pre  = float((annual_trips_B_fp * x_opt).sum())
fhwa_year_total_B   = float(sum(FHWA_VMT_B.values()))
print(f"[Annual] Pre-norm: Model = {annual_model_B_pre:.3f} B vs FHWA = {fhwa_year_total_B:.1f} B  "
      f"-> Error = {100*(annual_model_B_pre-fhwa_year_total_B)/fhwa_year_total_B:.2f}%")
if NORMALIZE_TO_FHWA:
    annual_model_B_post = float(vmt_miles.sum() / 1e9)
    print(f"[Annual] Post-norm: Model = {annual_model_B_post:.3f} B vs FHWA = {fhwa_year_total_B:.1f} B  "
          f"-> Error = {100*(annual_model_B_post-fhwa_year_total_B)/fhwa_year_total_B:.4f}%")


# ---------------- PART 2: Base EV allocation (round-trip; no mid-trip charging) ----------------
def ev_ratio_per_bin(R, x_vec, round_trip: bool = True):
    """EV fraction per bin for range R.
       One-way:     min(1, R / d_i)
       Round-trip:  min(1, R / (2*d_i))   (base requirement)
    """
    d = np.array(x_vec, float)
    factor = 2.0 if round_trip else 1.0
    return np.minimum(1.0, R / np.maximum(factor * d, 1e-9))

def compute_split_for_R(R, round_trip: bool = True):
    ratio = ev_ratio_per_bin(R, x_opt, round_trip=round_trip)             # length B
    ev_miles_bins_monthly  = vmt_miles * ratio.reshape(1, -1)
    gas_miles_bins_monthly = vmt_miles - ev_miles_bins_monthly

    ev_annual_miles  = ev_miles_bins_monthly.sum()
    gas_annual_miles = gas_miles_bins_monthly.sum()
    total_annual_miles = ev_annual_miles + gas_annual_miles

    # Per-bin annual (B)
    bin_vmt_B = vmt_miles.sum(axis=0) / 1e9
    bin_ev_B  = ev_miles_bins_monthly.sum(axis=0) / 1e9
    bin_gas_B = gas_miles_bins_monthly.sum(axis=0) / 1e9

    per_bin = pd.DataFrame({
        "Trip Distance Bin (miles)": bins,
        "Average Distance (miles)": np.round(x_opt, 1),
        "Trips (B/year)": np.round(annual_trips_B_fp, 3),
        "VMT (B/year)": np.round(bin_vmt_B, 3),
        "EV VMT (B/year)": np.round(bin_ev_B, 3),
        "Gas VMT (B/year)": np.round(bin_gas_B, 3),
        "EV% in bin": np.round(100 * bin_ev_B / np.maximum(bin_vmt_B, 1e-12), 1),
    }).sort_values("Trip Distance Bin (miles)", key=lambda s: s.map(lb)).reset_index(drop=True)

    return {
        "R": R,
        "EV_VMT_B": ev_annual_miles / 1e9,
        "Gas_VMT_B": gas_annual_miles / 1e9,
        "EV_share_pct": 100 * ev_annual_miles / np.maximum(total_annual_miles, 1e-12),
        "per_bin": per_bin
    }

# Example: 50 miles (base)
res50 = compute_split_for_R(50, round_trip=ROUND_TRIP)
display(res50["per_bin"])
print(f"s(50) EV share (round-trip) = {res50['EV_share_pct']:.2f}% | "
      f"EV VMT = {res50['EV_VMT_B']:.2f} B, Gas VMT = {res50['Gas_VMT_B']:.2f} B")

# ---------------- PART 2B: Emissions + battery/cost metrics by range ----------------
gas_g_per_mile = CO2_PER_GAL / MPG_GAS
ev_g_per_mile  = EV_KWH_PER_MI * GRID_G_PER_KWH
delta_g_per_mi = gas_g_per_mile - ev_g_per_mile

fhwa_year_total_miles = fhwa_year_total_B * 1e9
n_vehicles = fhwa_year_total_miles / AVG_DRIVER_MILES
def pack_kwh(R): return R / ETA_MI_PER_KWH

rows = []
for R in RANGES:
    res = compute_split_for_R(R, round_trip=ROUND_TRIP)
    ev_miles = res["EV_VMT_B"] * 1e9

    size_kwh = pack_kwh(R)
    installed_kwh = n_vehicles * size_kwh
    installed_TWh = installed_kwh / 1e9    # ✅ true TWh
    co2_saved_tons = (ev_miles * delta_g_per_mi) / 1e6
    fleet_cost = installed_kwh * PACK_COST_PER_KWH

    rows.append({
        "Range (mi)": R,
        "EV share (%)": res["EV_share_pct"],
        "EV VMT (B)": res["EV_VMT_B"],
        "Gas VMT (B)": res["Gas_VMT_B"],
        "EV CO2 saved (Mt)": co2_saved_tons / 1e6,
        "Battery size (kWh)": size_kwh,
        "Installed battery (TWh)": installed_TWh,
        "kWh battery / electric mile": installed_kwh / max(ev_miles, 1.0),
        "kWh battery / ton CO2 saved": installed_kwh / max(co2_saved_tons, 1e-9),
        "Fleet pack cost ($B)": fleet_cost / 1e9,
        "$ / electric mile": fleet_cost / max(ev_miles, 1.0),
        "$ / ton CO2 saved": fleet_cost / max(co2_saved_tons, 1e-9),
    })

summary_base = pd.DataFrame(rows).sort_values("Range (mi)").reset_index(drop=True)
display(summary_base)
summary_base.to_csv(f"{OUTDIR}/range_summary_metrics_RT_base.csv", index=False)

# ---------------- PART 3: Simple classic plots (autoscaled) ----------------
xs = summary_base["Range (mi)"].to_numpy()
def savefig(path):
    plt.tight_layout(); plt.savefig(path, dpi=200); plt.close()

plt.figure(); plt.plot(xs, summary_base["$ / ton CO2 saved"], marker="o")
plt.xlabel("EV range (miles)"); plt.ylabel("Cost per ton CO$_2$ saved ($/ton)")
plt.title("Cost per ton of CO$_2$ saved vs EV range"); savefig(f"{OUTDIR}/Figure_A10_cost_per_ton.png")

plt.figure(); plt.plot(xs, summary_base["$ / electric mile"], marker="o")
plt.xlabel("EV range (miles)"); plt.ylabel("Cost per electric mile ($/mile)")
plt.title("Cost per electric mile vs EV range"); savefig(f"{OUTDIR}/Figure_A11_cost_per_mile.png")

plt.figure(); plt.plot(xs, summary_base["EV VMT (B)"], marker="o", label="EV VMT (B)")
plt.plot(xs, summary_base["Gas VMT (B)"], marker="o", label="Gas VMT (B)")
plt.xlabel("EV range (miles)"); plt.ylabel("VMT (billion miles)")
plt.title("Electric vs. Gasoline VMT by EV Range (annual)"); plt.legend()
savefig(f"{OUTDIR}/ev_vs_gas_line.png")

plt.figure()
ev = summary_base["EV VMT (B)"].to_numpy(); gas = summary_base["Gas VMT (B)"].to_numpy()
plt.bar(xs, gas, label="Gas VMT (B)"); plt.bar(xs, ev, bottom=gas, label="EV VMT (B)")
plt.xlabel("EV range (miles)"); plt.ylabel("VMT (billion miles)")
plt.title("VMT Split by EV Range"); plt.legend(); savefig(f"{OUTDIR}/ev_vmt_bars.png")

plt.figure(); plt.plot(xs, summary_base["EV CO2 saved (Mt)"], marker="o")
plt.xlabel("EV range (miles)"); plt.ylabel("CO$_2$ saved (Mt)")
plt.title("Annual CO$_2$ Savings vs EV Range"); savefig(f"{OUTDIR}/carbon_savings_vs_range.png")

plt.figure(); plt.plot(xs, summary_base["kWh battery / electric mile"], marker="o")
plt.xlabel("EV range (miles)"); plt.ylabel("kWh of battery per electric mile")
plt.title("Installed Battery Intensity vs EV Range"); savefig(f"{OUTDIR}/kwh_per_electric_mile.png")

# ---------------- PART 4: Scenarios — charges/week from your doc + ACS weights ----------------
# Level ladder & charges/week (per your notes)
LEVEL_CPW = {
    "everyday": 7.0,
    "garage":   6.0,   # Owner Occ 1, detached
    "driveway": 5.0,   # Renter 1, detached; Owner 1, attached; Owner 2-4
    "work":     3.0,   # Renter 1, attached; Owner 5-49
    "weekly":   1.0,   # Renter 2-4; Owner 50+
    "errands":  0.5,   # Additional level from note (optional mapping)
    "none":     0.0,   # Additional level from note (optional mapping)
}
LEVEL_ORDER = ["none", "errands", "weekly", "work", "driveway", "garage", "everyday"]

def shift_level(level_name: str, delta: int) -> str:
    i = max(0, min(len(LEVEL_ORDER)-1, LEVEL_ORDER.index(level_name) + delta))
    return LEVEL_ORDER[i]

def acs_level_weights_from_B25032():
    """Derive national weights over charging *levels* from ACS B25032 (Tenure x Units)."""
    try:
        meta = get_json(ACS_VARS_URL)
        vars_meta = meta.get("variables", {})
    except Exception as e:
        warnings.warn(f"ACS metadata fetch failed: {e}. Falling back to default weights.")
        return {"garage": .45, "driveway": .25, "work": .20, "weekly": .10}  # simple fallback

    def find_vars(owner=True, pattern="1, detached"):
        side = "Owner occupied" if owner else "Renter occupied"
        out = []
        for k, v in vars_meta.items():
            if not k.startswith(ACS_TABLE + "_"):
                continue
            lab = v.get("label", "")
            if side in lab and pattern in lab and ("!!Estimate" in lab):
                out.append(k)
        return out

    # Owner occupied groups
    v_owner_1_det = find_vars(owner=True,  pattern="1, detached")
    v_owner_1_att = find_vars(owner=True,  pattern="1, attached")
    v_owner_2apt  = find_vars(owner=True,  pattern="2 apartments")
    v_owner_3_4   = find_vars(owner=True,  pattern="3 or 4 apartments")
    v_owner_5_9   = find_vars(owner=True,  pattern="5 to 9")
    v_owner_10_19 = find_vars(owner=True,  pattern="10 to 19")
    v_owner_20_49 = find_vars(owner=True,  pattern="20 to 49")
    v_owner_50p   = find_vars(owner=True,  pattern="50 or more")

    # Renter occupied groups
    v_renter_1_det = find_vars(owner=False, pattern="1, detached")
    v_renter_1_att = find_vars(owner=False, pattern="1, attached")
    v_renter_2apt  = find_vars(owner=False, pattern="2 apartments")
    v_renter_3_4   = find_vars(owner=False, pattern="3 or 4 apartments")
    v_renter_5_9   = find_vars(owner=False, pattern="5 to 9")
    v_renter_10_19 = find_vars(owner=False, pattern="10 to 19")
    v_renter_20_49 = find_vars(owner=False, pattern="20 to 49")
    v_renter_50p   = find_vars(owner=False, pattern="50 or more")

    ALL_VARS = sorted(set(
        v_owner_1_det + v_owner_1_att + v_owner_2apt + v_owner_3_4 + v_owner_5_9 + v_owner_10_19 + v_owner_20_49 + v_owner_50p +
        v_renter_1_det + v_renter_1_att + v_renter_2apt + v_renter_3_4 + v_renter_5_9 + v_renter_10_19 + v_renter_20_49 + v_renter_50p
    ))
    if not ALL_VARS:
        warnings.warn("Could not locate B25032 variable IDs. Falling back to default weights.")
        return {"garage": .45, "driveway": .25, "work": .20, "weekly": .10}

    try:
        params = {"get": "NAME," + ",".join(ALL_VARS), "for": "us:1"}
        data = get_json(ACS_DATA_URL, params=params)
        header, values = data[0], data[1]
        rec = dict(zip(header, values))
    except Exception as e:
        warnings.warn(f"ACS data fetch failed: {e}. Falling back to default weights.")
        return {"garage": .45, "driveway": .25, "work": .20, "weekly": .10}

    def ssum(vlist):
        return sum(float(rec[v]) for v in vlist if v in rec and rec[v] not in (None, "", "null"))

    w_by_level = {k: 0.0 for k in LEVEL_CPW.keys()}

    # garage (6): Owner 1, detached
    w_by_level["garage"] += ssum(v_owner_1_det)

    # driveway (5): Renter 1, detached; Owner 1, attached; Owner 2-4
    w_by_level["driveway"] += ssum(v_renter_1_det)
    w_by_level["driveway"] += ssum(v_owner_1_att)
    w_by_level["driveway"] += ssum(v_owner_2apt + v_owner_3_4)

    # work (3): Renter 1, attached; Owner 5-49
    w_by_level["work"] += ssum(v_renter_1_att)
    w_by_level["work"] += ssum(v_owner_5_9 + v_owner_10_19 + v_owner_20_49)

    # weekly (1): Renter 2-4; Owner 50+
    w_by_level["weekly"] += ssum(v_renter_2apt + v_renter_3_4)
    w_by_level["weekly"] += ssum(v_owner_50p)

    # Optional: include renter 5-49 & renter 50+ if desired; uncomment next line
    # w_by_level["weekly"] += ssum(v_renter_5_9 + v_renter_10_19 + v_renter_20_49 + v_renter_50p)

    # Normalize to sum 1.0 over populated levels
    tot = sum(w_by_level.values())
    if tot > 0:
        for k in list(w_by_level.keys()):
            w_by_level[k] = w_by_level[k] / tot

    # Remove zero-weight levels to avoid confusion
    w_by_level = {k: v for k, v in w_by_level.items() if v > 0}
    return w_by_level

def level_weights():
    if not USE_ACS and LEVEL_WEIGHT_OVERRIDE:
        s = sum(LEVEL_WEIGHT_OVERRIDE.values())
        if abs(s - 1.0) > 1e-6:
            raise ValueError("LEVEL_WEIGHT_OVERRIDE must sum to 1.0")
        return dict(LEVEL_WEIGHT_OVERRIDE)
    return acs_level_weights_from_B25032()

def cpw_from_level_weights(wlevels: dict):
    return sum(LEVEL_CPW[k] * w for k, w in wlevels.items())

def shift_weights(wlevels: dict, delta: int):
    out = {k: 0.0 for k in LEVEL_CPW.keys()}
    for lvl, w in wlevels.items():
        new_lvl = shift_level(lvl, delta)
        out[new_lvl] += w
    # drop zeros
    return {k: v for k, v in out.items() if v > 0}

BASE_SCENARIOS = {
    "Base_EachTrip": "base",  # unlimited (each trip starts full)
    "Low":           "low",
    "Normal":        "normal",
    "More":          "more",
}

def scenario_cpw(scen_name: str) -> float:
    if scen_name == "Base_EachTrip":
        return math.inf
    w_norm = level_weights()
    if scen_name == "Normal":
        W = w_norm
    elif scen_name == "Low":
        W = shift_weights(w_norm, delta=-1)   # everything down 1 level
    elif scen_name == "More":
        W = shift_weights(w_norm, delta=+1)   # everything up 1 level
    else:
        raise ValueError(f"Unknown scenario: {scen_name}")
    return cpw_from_level_weights(W)

def compute_with_weekly_budget(R, scen_name: str):
    cpw = scenario_cpw(scen_name)
    # Base: unlimited charges -> same as base split
    if math.isinf(cpw):
        return compute_split_for_R(R, round_trip=ROUND_TRIP)

    # Fleet-wide annual EV-mile budget
    weekly_ev_cap_per_vehicle = cpw * R                 # miles/week/vehicle
    fleet_budget_ev_miles = weekly_ev_cap_per_vehicle * 52 * n_vehicles

    # Base (unconstrained) EV miles by bin (annual)
    ratio_base = ev_ratio_per_bin(R, x_opt, round_trip=ROUND_TRIP)        # length B
    ev_miles_bins_monthly  = vmt_miles * ratio_base.reshape(1, -1)
    base_ev_bins_annual    = ev_miles_bins_monthly.sum(axis=0)            # miles
    base_ev_total_annual   = base_ev_bins_annual.sum()

    if base_ev_total_annual <= fleet_budget_ev_miles:
        return compute_split_for_R(R, round_trip=ROUND_TRIP)

    # Budget binding → scale EV miles proportionally across bins
    scale = fleet_budget_ev_miles / base_ev_total_annual
    ev_bins_annual = base_ev_bins_annual * scale
    gas_bins_annual = vmt_miles.sum(axis=0) - ev_bins_annual

    res = {
        "R": R,
        "EV_VMT_B": ev_bins_annual.sum() / 1e9,
        "Gas_VMT_B": gas_bins_annual.sum() / 1e9,
        "EV_share_pct": 100 * ev_bins_annual.sum() / np.maximum((ev_bins_annual+gas_bins_annual).sum(), 1e-12),
        "per_bin": pd.DataFrame({
            "Trip Distance Bin (miles)": bins,
            "Average Distance (miles)": np.round(x_opt, 1),
            "Trips (B/year)": np.round(annual_trips_B_fp, 3),
            "VMT (B/year)": np.round((vmt_miles.sum(axis=0)/1e9), 3),
            "EV VMT (B/year)": np.round(ev_bins_annual/1e9, 3),
            "Gas VMT (B/year)": np.round(gas_bins_annual/1e9, 3),
            "EV% in bin": np.round(100 * (ev_bins_annual/np.maximum(vmt_miles.sum(axis=0),1e-12)), 1),
        }).sort_values("Trip Distance Bin (miles)", key=lambda s: s.map(lb)).reset_index(drop=True)
    }
    return res

# -------- Example scenario run (uncomment to use) --------
for scen in ["Low", "Normal", "More"]:
    rows = []
    for R in RANGES:
        res = compute_with_weekly_budget(R, scen)
        rows.append({"Range (mi)": R, "Scenario": scen,
                     "EV VMT (B)": res["EV_VMT_B"],
                     "EV share (%)": res["EV_share_pct"]})
    df = pd.DataFrame(rows)
    display(df)
    df.to_csv(f"{OUTDIR}/summary_{scen}.csv", index=False)


,Month,FHWA VMT (B),Model VMT (B) (pre-norm),Error (%) (pre-norm)
0,1,248.9,278.367,11.84
1,2,235.4,260.225,10.55
2,3,273.7,286.381,4.63
3,4,258.2,280.000,8.44
4,5,289.6,283.511,-2.10
5,6,285.6,274.953,-3.73
6,7,289.8,294.999,1.79
7,8,290.9,290.024,-0.30
8,9,278.0,284.651,2.39
9,10,284.0,288.695,1.65


Max |Error| % (pre-normalization): 11.84
Max |Error| % (post-normalization): 0.0


,Trip Distance Bin (miles),Trips (B/month),Calculated Average Bin Distance (miles),Calculated VMT (B)
0,0–1,12.173,0.3,3.652
1,1–3,10.221,1.0,10.221
2,3–5,5.311,3.0,15.933
3,5–10,6.909,5.0,34.545
4,10–25,6.668,10.0,66.680
5,25–50,2.074,25.0,51.850
6,50–100,0.682,50.0,34.100
7,100–250,0.304,100.0,30.400
8,250–500,0.062,250.0,15.500
9,500+,0.044,500.0,22.000


[Month 12] Pre-norm: VMT = 284.078 B vs FHWA = 263.6 B  -> Error = 7.77% | Post-norm: 263.600 B (exact FHWA)


,Trip Distance Bin (miles),Trips (B/year),Average Bin Distance (miles),Calculated VMT (B/year)
0,0–1,146.472,0.3,43.942
1,1–3,123.019,1.0,123.019
2,3–5,63.632,3.0,190.896
3,5–10,81.916,5.0,409.580
4,10–25,77.731,10.0,777.310
5,25–50,24.428,25.0,610.700
6,50–100,8.245,50.0,412.250
7,100–250,3.748,100.0,374.800
8,250–500,0.774,250.0,193.500
9,500+,0.513,500.0,256.500


...

,Range (mi),Scenario,EV VMT (B),EV share (%)
0,50,Low,2042.876408,62.611144
1,75,Low,2551.008080,78.184629
2,100,Low,2710.947431,83.086534
3,125,Low,2771.575362,84.944691
4,150,Low,2832.203294,86.802847


/tmp/ipython-input-1407596679.py:484: UserWarning: Could not locate B25032 variable IDs. Falling back to default weights.
  warnings.warn("Could not locate B25032 variable IDs. Falling back to default weights.")


,Range (mi),Scenario,EV VMT (B),EV share (%)
0,50,Normal,2391.068730,73.282724
1,75,Normal,2551.008080,78.184629
2,100,Normal,2710.947431,83.086534
3,125,Normal,2771.575362,84.944691
4,150,Normal,2832.203294,86.802847


/tmp/ipython-input-1407596679.py:484: UserWarning: Could not locate B25032 variable IDs. Falling back to default weights.
  warnings.warn("Could not locate B25032 variable IDs. Falling back to default weights.")


,Range (mi),Scenario,EV VMT (B),EV share (%)
0,50,More,2391.068730,73.282724
1,75,More,2551.008080,78.184629
2,100,More,2710.947431,83.086534
3,125,More,2771.575362,84.944691
4,150,More,2832.203294,86.802847
